In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from onomatron.data import Vocab, load_names, make_ngram_pairs
from onomatron.linear import LinearBigram
from onomatron.ngram import NGramModel

%matplotlib inline

# 3 — Simple neural net: the linear bigram

Goal: match the count-based bigram score with a model trained by gradient
descent. 

One linear layer over one-hot contexts,
`logits = xenc @ W`, trained with cross-entropy on the next character.


## Load names and build the vocab


In [ ]:
names = load_names("../data/processed/names.txt")
vocab = Vocab(names)
print(f"{len(names)} names, vocab of {len(vocab)} chars")

## Bigrams -> tensors


In [ ]:
pairs = make_ngram_pairs(names, vocab, n=2)  # (N, 2): context char, target char
xs, ys = pairs[:, 0], pairs[:, 1]
xenc = F.one_hot(xs, num_classes=len(vocab)).float()
print(f"{xs.nelement()} bigrams, xenc {tuple(xenc.shape)}")

## LR range test

One fresh model per learning rate (fixed seed), 100 SGD steps each, then plot
the final loss on a log scale. We keep a decade below the minimum.


In [ ]:
def lr_range_test(lrs: list[float], steps: int = 100, seed: int = 42) -> list[float]:
    """Final loss after `steps` SGD steps at each lr, model re-seeded each time."""
    losses = []
    for lr in lrs:
        model = LinearBigram(vocab, seed=seed)
        opt = torch.optim.SGD(model.parameters(), lr=lr)
        for _ in range(steps):
            loss = model.loss(xenc, ys)
            opt.zero_grad()
            loss.backward()
            opt.step()
        losses.append(loss.item())
    return losses


lrs = (10 ** torch.linspace(-3, 3, 100)).tolist()  # 1e-3 -> 1e3, log scale
losses = lr_range_test(lrs)

plt.plot(lrs, losses)
plt.xscale("log")
plt.xlabel("learning rate")
plt.ylabel("loss after 100 steps")

best_lr = lrs[int(torch.tensor(losses).argmin())]
lr = best_lr / 10  # one decade below the minimum, safety margin
print(f"minimum at lr={best_lr:.4f} -> using lr={lr:.4f}")

## Train with the chosen lr

In [ ]:
model = LinearBigram(vocab, seed=42)
opt = torch.optim.SGD(model.parameters(), lr=lr)

losses = []
for _ in range(5000):
    loss = model.loss(xenc, ys)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")

bigram = NGramModel.fit(names, vocab, n=2)
trigram = NGramModel.fit(names, vocab, n=3)
print(f"NLL linear NN : {model.nll(xenc, ys):.4f}")
print(f"NLL bigram    : {bigram.nll(names):.4f}")
print(f"Nll trigram  : {trigram.nll(names):.4f}")

## Sample with single Layer NN

In [ ]:
model.sample(n=20, seed=42)

In [ ]:
bigram.sample(20, seed=42)